# Predictive Maintenance — Exploratory Analysis

IoT sensor data exploration, LSTM/GRU failure prediction, and anomaly detection.

Run `make setup` first, then select kernel **Predictive Maintenance (venv)**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.data.generate_synthetic_data import generate_dataset, save_dataset
from src.data.preprocessing import run_preprocessing, SENSOR_COLS
from src.anomaly.rolling_window import detect_multivariate_anomalies

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
with open(ROOT / "config/config.yaml") as f:
    config = yaml.safe_load(f)

df = generate_dataset(config)
save_dataset(df, config["data"]["raw_dir"])
df.head()

In [ ]:
sample = df[df["machine_id"] == 0]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axes, SENSOR_COLS):
    ax.plot(sample["timestamp"], sample[col], label=col)
    ax.set_ylabel(col)
    failure_region = sample[sample["failure"] == 1]
    if len(failure_region):
        ax.axvspan(failure_region["timestamp"].iloc[0], failure_region["timestamp"].iloc[-1], alpha=0.2, color="red", label="failure")
axes[0].legend()
plt.suptitle("Machine 0 — Sensor Degradation Before Failure")
plt.tight_layout()

In [ ]:
anomaly_df = detect_multivariate_anomalies(df, SENSOR_COLS, window=30, threshold=3.0)
print(f"Rolling anomaly rate: {anomaly_df['rolling_anomaly'].mean():.2%}")
anomaly_df.groupby('machine_id')['rolling_anomaly'].mean().hist(bins=20)
plt.title("Per-Machine Anomaly Rate Distribution")
plt.xlabel("Anomaly rate")

In [ ]:
# Full training pipeline
!python {ROOT}/pipelines/local_train.py --model all